processing pipeline for LSH implementation of dictionary: 
1) extract pose info
2) process - flatten frames, grab pseudo-keyframes, etc.

In [38]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
# import mediapipe.python.solutions.hands as mp_hands
# import mediapipe.python.solutions
import cv2 as cv
import os
import shutil
import sys
import numpy as np
from scipy import stats
import pandas as pd
from pathlib import Path
np.set_printoptions(threshold=sys.maxsize) # prevents long arrays from saving to csv with "..."


In [39]:
ROOT = Path.cwd()

In [40]:
# function for extracting body pose coordinates

def get_pose_coords(input_vid, pose_model_path):

    BaseOptions = mp.tasks.BaseOptions
    PoseLandmarker = mp.tasks.vision.PoseLandmarker
    PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
    VisionRunningMode = mp.tasks.vision.RunningMode

    # Create a pose landmarker instance with the video mode:
    options = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=pose_model_path),
        running_mode=VisionRunningMode.VIDEO)

    with PoseLandmarker.create_from_options(options) as landmarker:
    # The landmarker is initialized. Use it here.
        options=mp.tasks.vision.PoseLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path='pose_landmarker.task'),
        running_mode=mp.tasks.vision.RunningMode.VIDEO, # Set to video mode (not image or livestream)
    )
    
    # Initialize MediaPipe Pose
    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose(static_image_mode=False, 
                    min_detection_confidence=0.5, 
                    min_tracking_confidence=0.5)
    # mp_drawing = mp.solutions.drawing_utils

    # Open the video file
    cap = cv.VideoCapture(input_vid)

    if not cap.isOpened():
        print("Error: Could not open video.")
        exit()

    # Handle result files!
    result_file_name = str(input_vid)[:-4] + "_pose-results.txt" # cut off .mp4 and add _pose-results.txt !
    # results_file = open(result_file_name, "w")
    # ROOT = Path.cwd()

    pose_directory_path = ROOT / "asl_citation_forms" / "pose_results"
    # pose_directory_path = "/Users/ashlynwinship/Desktop/reverse-dictionary/asl_citation_forms/pose_results"
    # Create the directory if it doesn't exist (parents=True creates parent directories if needed)
    os.makedirs(pose_directory_path, exist_ok=True)
    # Construct the full file path
    result_file_path = os.path.join(pose_directory_path, result_file_name)
    # Open the file in write mode ('w') and write some content
    results_file = open(result_file_path, "w")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        # Convert the BGR image to RGB
        image_rgb = cv.cvtColor(frame, cv.COLOR_BGR2RGB)

        # Process the image and detect pose landmarks
        results = pose.process(image_rgb)

        # Write pose landmarks to file
        if results.pose_landmarks:
            for id, lm in enumerate(results.pose_landmarks.landmark):
                # Get image dimensions to convert normalized coordinates to pixel coordinates
                h, w, c = frame.shape
                cx, cy = int(lm.x * w), int(lm.y * h)
                # print(f"Landmark {id}: x={cx}, y={cy}, z={lm.z}, visibility={lm.visibility}")
                # results_file.write(f"Landmark {id}: x={cx}, y={cy}, z={lm.z}, visibility={lm.visibility}\n")
                results_file.write(f"{id}, {cx}, {cy}, {lm.z}\n")

    # Release resources
    cap.release()
    cv.destroyAllWindows()
    cv.waitKey(1)           # MacBook fix - need to add this for the window to actually close!

    results_file.close() # close file!
    
    return results_file

In [41]:
# Set the model

pose_model_path = ROOT / "pose_landmarker_lite.task"
# pose_model_path = '/Users/ashlynwinship/Desktop/reverse-dictionary/pose_landmarker_lite.task'

In [42]:
# Specify videos to extract pose coords from ("training" data)

folder_path = ROOT / "asl_citation_forms"
# folder_path = '/Users/ashlynwinship/Desktop/reverse-dictionary/asl_citation_forms' 

for filename in os.listdir(folder_path):
    # if ".webm" in filename: # for semlex
    if ".mp4" in filename:
        file_path = os.path.join(folder_path, filename)
        get_pose_coords(file_path, pose_model_path)

Context leak detected, CoreAnalytics returned false
I0000 00:00:1780494798.827280  102337 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M5
W0000 00:00:1780494798.880817  114488 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1780494798.885050  114488 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1780494798.892052  102337 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M5
W0000 00:00:1780494798.934601  114496 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1780494798.940184  114496 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tenso

In [ ]:
# Sort training results into subdirectories!

pose_path = ROOT / "asl_citation_forms" / "pose_results"
# pose_path = '/Users/ashlynwinship/Desktop/reverse-dictionary/asl_citation_forms/pose_results'

for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)
    if "pose" in filename:
        dest = pose_path / filename
        if dest.exists(): # remove and replace if it already exists!
            dest.unlink()
        shutil.move(file_path, pose_path)

Error: Destination path '/Users/ashlynwinship/Desktop/asl_summer26/data_processing/asl_citation_forms/pose_results/and_pose-results.txt' already exists

In [ ]:
# Get pose coordinates for my test chocolate video

test_vid = ROOT / "chocolate-me.mov"
# print(test_vid)
get_pose_coords(test_vid, pose_model_path)

/Users/ashlynwinship/Desktop/asl_summer26/data_processing/chocolate-me.mov


I0000 00:00:1780494543.864474  102337 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M5
W0000 00:00:1780494543.926735  110247 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1780494543.931999  110247 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1780494543.939276  102337 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M5


TypeError: 'PosixPath' object is not subscriptable

W0000 00:00:1780494543.985218  110256 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1780494543.992037  110256 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [ ]:
# Import ASL LEX data (replace with the new dataset!)

asl_lex_data = pd.read_csv("asl_lex.csv")
# columns to keep: LemmaID, Handshape.2.0, SignType.2.0, Movement.2.0, RepeatedMovement.2.0, MajorLocation.2.0, MinorLocation.2.0, SecondMinorLocation.2.0, Contact.2.0, NonDominantHandshape.2.0
asl_lex_data = asl_lex_data[['LemmaID', 'Handshape.2.0', 'SignType.2.0', 'Movement.2.0', 'RepeatedMovement.2.0', 'MajorLocation.2.0', 'MinorLocation.2.0', 'SecondMinorLocation.2.0', 'Contact.2.0', 'NonDominantHandshape.2.0']]
asl_lex_data = asl_lex_data.rename(columns={'Handshape.2.0':"Handshape", 'SignType.2.0': 'SignType', 'Movement.2.0':'Movement', 'RepeatedMovement.2.0':'RepeatedMovement', 'MajorLocation.2.0':'MajorLocation', 'MinorLocation.2.0':'MinorLocation', 'SecondMinorLocation.2.0':'MinorLocation2', 'Contact.2.0':'Contact', 'NonDominantHandshape.2.0':'NonDomHandshape'})
asl_lex_data = asl_lex_data.drop(asl_lex_data.index[0]) # drop this - csv had a double-header row with no relevant info in it
asl_lex_data.head()

# # Import semlex data
# semlex_data = pd.read_csv('/Users/ashlynwinship/Desktop/asl-comp-seminar/reverse-dictionary/semlex_metadata.csv') # local

# semlex_data = semlex_data[['video_id', "split", "label", "Handshape", "SignType", "PathMovement", "RepeatedMovement", "MajorLocation", "Contact", "NondominantHandshape"]]
# semlex_data = semlex_data[semlex_data['split'] == 'train']

,LemmaID,Handshape,SignType,Movement,RepeatedMovement,MajorLocation,MinorLocation,MinorLocation2,Contact,NonDomHandshape
1,tree,5,OneHanded,Zero,1.0,Neutral,Neutral,NaN,0.0,NaN
2,night,flat_b,AsymmetricalSameHandshape,Straight,1.0,Hand,PalmBack,NaN,1.0,flat_b
3,hamburger,c,SymmetricalOrAlternating,Straight,0.0,Neutral,Neutral,Neutral,1.0,c
4,nephew,flat_n,OneHanded,Zero,1.0,Head,HeadAway,NaN,0.0,NaN
5,castle,curved_v,SymmetricalOrAlternating,Straight,1.0,Neutral,Neutral,Neutral,1.0,curved_v


In [ ]:
# Import pose results

def load_pose_data(input_file, pose_dict):
    data = []
    with open(input_file, "r") as file:
        one_frame = []
        for line in file:
            line = line.strip()
            line = line.split(", ")
            for i in range(len(line)):
                line[i] = float(line[i])
            # print(line)
            if line[0] > 0:
                one_frame.append([line[1], line[2], line[3]])
            elif line[0] == 0:
                if len(one_frame) != 0:
                    data.append(one_frame) # make sure to skip the first empty one_frame since the first index is 0!
                one_frame = [] # reset
                one_frame.append([line[1], line[2], line[3]])
        # print(one_frame)
    # print(data)
    data = np.array(data) # change list of lists to array!
    # key_str = input_file[97:-17] # chop out file path to only include the lemma!
    key_str = Path(input_file).stem.replace("_pose-results", "")
    if key_str not in pose_dict.keys():
        pose_dict[key_str] = data
    return pose_dict

# sanity check!
# pose_dict = {}
# load_pose_data('/Users/ashlynwinship/Desktop/asl-comp-seminar/reverse-dictionary/asl_citation_forms/pose_results/and_pose-results.txt', pose_dict)
# print(len(pose_dict['and'][0]))

folder_path = ROOT / "asl_citation_forms" / "pose_results"
# folder_path = '/Users/ashlynwinship/Desktop/reverse-dictionary/asl_citation_forms/pose_results' 

pose_dict = {}
for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)
    # print(file_path)
    if file_path == ROOT / "asl_citation_forms" / "pose_results" / ".DS_Store": # no idea what this file is but it's not a citation form and causes utf-8 problems lol
    # if file_path == '/Users/ashlynwinship/Desktop/reverse-dictionary/asl_citation_forms/pose_results/.DS_Store': # no idea what this file is but it's not a citation form and causes utf-8 problems lol
        pass
    else:
        load_pose_data(file_path, pose_dict)

# pose_dict

In [ ]:
# Add pose landmark arrays to SEMLEX or ASL LEX data

asl_lex_data['pose_landmarks'] = ""

for i, row in asl_lex_data.iterrows():
    lemma = row['LemmaID'] # 'label' for semlex
    if lemma in pose_dict.keys():
        asl_lex_data.at[i, 'pose_landmarks'] = pose_dict[lemma]

# asl_lex_data.head()

In [ ]:
# Filter asl_lex to only those lemmas with landmarks!

filtered_asl_lex = asl_lex_data.loc[asl_lex_data['LemmaID'].isin(pose_dict.keys())]
filtered_asl_lex.head()

,LemmaID,Handshape,SignType,Movement,RepeatedMovement,MajorLocation,MinorLocation,MinorLocation2,Contact,NonDomHandshape,pose_landmarks
3,hamburger,c,SymmetricalOrAlternating,Straight,0.0,Neutral,Neutral,Neutral,1.0,c,"[[[989.0, 275.0, -0.7995476126670837], [1018.0..."
7,cup,c,AsymmetricalDifferentHandshape,Straight,1.0,Hand,Palm,NaN,1.0,B,"[[[973.0, 241.0, -0.5753063559532166], [1001.0..."
8,english,c,AsymmetricalSameHandshape,Straight,1.0,Hand,PalmBack,NaN,1.0,c,"[[[1007.0, 250.0, -0.7349555492401123], [1040...."
11,dentist,s,OneHanded,Straight,1.0,Head,Mouth,NaN,1.0,NaN,"[[[1037.0, 278.0, -0.7727239727973938], [1063...."
14,chair,h,DominanceViolation,Straight,1.0,Hand,FingerBack,NaN,1.0,DominanceConditionViolation,"[[[1032.0, 242.0, -0.8620423674583435], [1062...."


In [ ]:
# Filter pose_landmarks:
# 1) First 10 frames
# 2) Limited pseudo-keyframes (every 10 frames, up to 7 frames total)
# 3) All pseudo-keyframes (every 10 frames)

filtered_asl_lex['pose_first10'] = ""
filtered_asl_lex['pose_pseudokey_n7'] = ""
filtered_asl_lex['pose_pseudokey_all'] = ""
first_ten_pose_frames = []
pseudokey_n7_frames = []
pseudokey_all_frames = []
for i, row in filtered_asl_lex.iterrows():
    all_frames = row['pose_landmarks']
    # print("all frames length: ", len(all_frames))

    ten_frames = [all_frames[0], all_frames[1], all_frames[2], all_frames[3], all_frames[4], all_frames[5], all_frames[6], all_frames[7], all_frames[8], all_frames[9]]
    pseudo_n7_frames = [all_frames[0], all_frames[10], all_frames[20], all_frames[30], all_frames[40], all_frames[50], all_frames[60]]
    pseudo_all_frames = []
    for i in range(0, len(all_frames), 10):
        current_frame = all_frames[i]
        # print(len(current_frame[0]))
        # print("current frame: ", current_frame)
        pseudo_all_frames.append(current_frame)

    first_ten_pose_frames.append(ten_frames)
    pseudokey_n7_frames.append(pseudo_n7_frames)
    pseudokey_all_frames.append(pseudo_all_frames)

filtered_asl_lex['pose_first10'] = first_ten_pose_frames
filtered_asl_lex['pose_pseudokey_n7'] = pseudokey_n7_frames
filtered_asl_lex['pose_pseudokey_all'] = pseudokey_all_frames
filtered_asl_lex.head()

# sanity check
# for i, row in filtered_asl_lex.iterrows():
#     print(len(row['pose_pseudokey_n7']))

/var/folders/kc/19sskvqd03q_7w6zpjtmzg0h0000gn/T/ipykernel_13845/1756384223.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_asl_lex['pose_first10'] = ""
/var/folders/kc/19sskvqd03q_7w6zpjtmzg0h0000gn/T/ipykernel_13845/1756384223.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_asl_lex['pose_pseudokey_n7'] = ""
/var/folders/kc/19sskvqd03q_7w6zpjtmzg0h0000gn/T/ipykernel_13845/1756384223.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
T

,LemmaID,Handshape,SignType,Movement,RepeatedMovement,MajorLocation,MinorLocation,MinorLocation2,Contact,NonDomHandshape,pose_landmarks,pose_first10,pose_pseudokey_n7,pose_pseudokey_all
3,hamburger,c,SymmetricalOrAlternating,Straight,0.0,Neutral,Neutral,Neutral,1.0,c,"[[[989.0, 275.0, -0.7995476126670837], [1018.0...","[[[989.0, 275.0, -0.7995476126670837], [1018.0...","[[[989.0, 275.0, -0.7995476126670837], [1018.0...","[[[989.0, 275.0, -0.7995476126670837], [1018.0..."
7,cup,c,AsymmetricalDifferentHandshape,Straight,1.0,Hand,Palm,NaN,1.0,B,"[[[973.0, 241.0, -0.5753063559532166], [1001.0...","[[[973.0, 241.0, -0.5753063559532166], [1001.0...","[[[973.0, 241.0, -0.5753063559532166], [1001.0...","[[[973.0, 241.0, -0.5753063559532166], [1001.0..."
8,english,c,AsymmetricalSameHandshape,Straight,1.0,Hand,PalmBack,NaN,1.0,c,"[[[1007.0, 250.0, -0.7349555492401123], [1040....","[[[1007.0, 250.0, -0.7349555492401123], [1040....","[[[1007.0, 250.0, -0.7349555492401123], [1040....","[[[1007.0, 250.0, -0.7349555492401123], [1040...."
11,dentist,s,OneHanded,Straight,1.0,Head,Mouth,NaN,1.0,NaN,"[[[1037.0, 278.0, -0.7727239727973938], [1063....","[[[1037.0, 278.0, -0.7727239727973938], [1063....","[[[1037.0, 278.0, -0.7727239727973938], [1063....","[[[1037.0, 278.0, -0.7727239727973938], [1063...."
14,chair,h,DominanceViolation,Straight,1.0,Hand,FingerBack,NaN,1.0,DominanceConditionViolation,"[[[1032.0, 242.0, -0.8620423674583435], [1062....","[[[1032.0, 242.0, -0.8620423674583435], [1062....","[[[1032.0, 242.0, -0.8620423674583435], [1062....","[[[1032.0, 242.0, -0.8620423674583435], [1062...."


In [ ]:
# Flatten!

# Round 1
filtered_asl_lex['flat_10frames'] = filtered_asl_lex['pose_first10'].apply(lambda x: np.array(x).flatten())
filtered_asl_lex['flat_pseudo_n7'] = filtered_asl_lex['pose_pseudokey_n7'].apply(lambda x: np.array(x).flatten())
filtered_asl_lex['flat_pseudo_all'] = filtered_asl_lex['pose_pseudokey_all'].apply(lambda x: np.array(x).flatten())

# Round 2
filtered_asl_lex['flat_10frames'] = filtered_asl_lex['flat_10frames'].apply(lambda x: x.flatten())
filtered_asl_lex['flat_pseudo_n7'] = filtered_asl_lex['flat_pseudo_n7'].apply(lambda x: x.flatten())
filtered_asl_lex['flat_pseudo_all'] = filtered_asl_lex['flat_pseudo_all'].apply(lambda x: x.flatten())

/var/folders/kc/19sskvqd03q_7w6zpjtmzg0h0000gn/T/ipykernel_13845/4088283184.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_asl_lex['flat_10frames'] = filtered_asl_lex['pose_first10'].apply(lambda x: np.array(x).flatten())
/var/folders/kc/19sskvqd03q_7w6zpjtmzg0h0000gn/T/ipykernel_13845/4088283184.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_asl_lex['flat_pseudo_n7'] = filtered_asl_lex['pose_pseudokey_n7'].apply(lambda x: np.array(x).flatten())
/var/folders/kc/19sskvqd03q_7w6zpj

In [ ]:
# Save to .csv, to be used in lsh_vid2vid.ipynb and lsh_frame2frame.ipynb
filtered_asl_lex.to_csv("frame_training_data.csv")